In [1]:
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
import urllib.parse
from sqlalchemy import create_engine,text,inspect
from sqlalchemy import Integer,String,Boolean,DateTime,Float
import datetime
import logging
import glob
import os
import sys

#________Logging Setup______________________________________________________________________________________

def setup_logging(log_file="UPI_Transactions_Pipeline.txt"):
    logger = logging.getLogger("UPITransactionPipeline")
    if logger.handlers:
        return logger

    logger.setLevel(logging.DEBUG)
    logger.propagate = False          # prevent bubbling to root logger

    formatter = logging.Formatter(
        fmt="%(asctime)s | %(levelname)-8s | %(name)s | %(message)s",
        datefmt="%Y-%m-%d %H:%M:%S"
    )

    console_handler = logging.StreamHandler(sys.stdout)
    console_handler.setLevel(logging.INFO)
    console_handler.setFormatter(formatter)

    file_handler = logging.FileHandler(log_file, mode='a', encoding='utf-8')
    # utf-8 encoding prevents crashes on Windows when logging
    # special characters like arrows or currency symbols
    file_handler.setLevel(logging.DEBUG)
    file_handler.setFormatter(formatter)

    logger.addHandler(console_handler)
    logger.addHandler(file_handler)
    return logger

# ______ Stage 1 : Ingestion ____________________________________________________________________________

def ingest(file_path, logger):
    logger.info("Stage 1 | Ingestion Started")
    logger.debug(f"Reading CSV From : {file_path}")

    if not os.path.exists(file_path):
        logger.error(f"File not Found : {file_path}")
        raise FileNotFoundError(f"File not Found : {file_path}")

    try:
        df = pd.read_csv(file_path)
        df.columns = df.columns.str.strip()   # remove accidental whitespace
        df = df.copy()                         # prevent SettingWithCopyWarning
        logger.info(f"Stage 1 | Ingestion Completed -> {df.shape[0]} rows, {df.shape[1]} columns")
        return df                              # returns DataFrame directly, not a dict
    except Exception as e:
        logger.error(f"Failed to read file: {e}")
        raise
#_________Stage 2: Remove Duplicates______________________________________________________________________________

def remove_duplicates(df,logger):
    before = df.shape[0]
    logger.info("Stage 2 | Cheking duplicates ")

    df = df.drop_duplicates()

    # DROP REASON: exact duplicate rows carry no new information and
    # would bias any aggregation or model trained on this data.

    dropped = before - df.shape[0]
    logger.info(f"Stage 2 | Duplicates removed : {dropped} ( rows remaining {df.shape[0]})")
    return df

#__________Stage 3 : Create Age Column__________________________________________________________________________

def clean_age(df,logger):
    before = df.shape[0]
    logger.info("Stage 3 | Creating Age column and removing outliers")

    current_year = datetime.datetime.now().year
    df['Age'] = current_year - df['Year_Birth']

    outliers = df[df['Age'] > 90]
    if not outliers.empty:
        logger.warning(f" Age > 90 found ({ len(outliers)} rows) : {outliers['Year_Birth'].tolist()}")

    df = df[df['Age'] <= 90]

    # DROP REASON: customers older than 90 imply Year_Birth values like 1893
    # which are clearly data-entry errors — no genuine supermarket customer
    # is 130 years old.

    logger.info(f"Stage 3 | Age range after cleaning :({df['Age'].min()} - {df['Age'].max()})")
    logger.info(f"Stage 3 | Rows dropped : {before - df.shape[0]} ( rows remaining {df.shape[0]})")
    return df

#_________Stage 4 : Clean Education Column_______________________________________________________________________

def clean_education(df,logger):
    logger.info("Stage 4 | Start Cleaning Education Column")
    logger.info(f"Before : { df['Education'].value_counts().to_dict()}")

    df['Education'] = df['Education'].replace({
        'Graduation' : 'Graduate',
        'PhD'        : 'Postgraduate',
        'Master'     : 'Postgraduate',
        '2n Cycle'   : 'Undergraduate',
        'Basic'      : 'Undergraduate'
    })
    # RENAME REASON: '2n Cycle' is ambiguous jargon; 'Basic' and '2n Cycle'
    # both represent pre-graduation education so they are merged into a single
    # 'Undergraduate' tier. PhD and Master are both postgraduate so merged too.
    # This gives 3 clean, interpretable tiers.

    logger.info(f"After : {df['Education'].value_counts().to_dict()}")
    logger.info("Stage 4 | Education Cleaning Complete")
    return df

#________Stage 5 : Clean Marital Status____________________________________________________________________________

def clean_Marital_Status(df,logger):
    logger.info("Stage 5 | Start Cleaning Marital_Status Column")
    logger.info(f"Before : {df['Marital_Status'].value_counts().to_dict()}")

    df['Marital_Status'] = df['Marital_Status'].replace({
    'Together' : 'Married',
    'Alone'    : 'Single',
    'YOLO'     : 'Single',
    'Absurd'   : 'Single'
    })
    # REMAP REASON:
    # 'Together'  -> 'Married'  : cohabiting partner = same household dynamic as married
    # 'Alone'     -> 'Single'   : synonym for Single, only 3 occurrences
    # 'YOLO','Absurd' -> 'Single': junk/test values (2 each); treated as not in a relationship
    # 'Divorced'  -> 'Single'   : no longer in a relationship — same marketing segment
    # 'Widow'     -> 'Single'   : no longer in a relationship — same marketing segment
    # Result: 2 clean groups (Married / Single) for campaign analysis

    logger.info(f"After : {df['Marital_Status'].value_counts().to_dict()}")
    logger.info("Stage 5 | Marital_Status cleaning complete")
    return df

#_______Stage 6 : Clean Income__________________________________________________________________________________________

def clean_income(df,logger):
    before = df.shape[0]
    logger.info("Stage 6 | Start Cleaning Income Column")

    # fill missing values :
    nulls = df['Income'].isnull().sum()
    if nulls > 0 :
        df['Income'].fillna(df['Income'].median() , inplace = True)
        # FILL REASON: median is used instead of mean because Income is right-skewed
        # (max = 666,666). The mean gets pulled upward by that extreme value, making
        # it unrepresentative of a typical customer. Median is robust to outliers.
        logger.warning(f'  Filled {nulls} missing Income values with median')

    # Remove outliers via IQR :
    Q1 = df['Income'].quantile(0.25)
    Q3 = df['Income'].quantile(0.75)
    IQR = Q3 - Q1
    upper = Q3 + 1.5 * IQR

    outliers = df[df['Income'] > upper]
    logger.warning(f"Income outliers (> {upper:.1f}) : {len(outliers)} rows")

    df = df[df['Income'] <= upper]
    # DROP REASON: rows with Income above the IQR upper fence (e.g. 666,666) are
    # clearly erroneous entries — not real customer incomes. These rows also show
    # near-zero spending which is inconsistent with high income, further confirming
    # they are bad data. Less than 0.4% of total rows removed.

    logger.info(f"Income Range after cleaning : ( {df['Income'].min()} - {df['Income'].max()})")
    logger.info(f"Stage 6 | Rows Dropped : ({ before - df.shape[0]}) ,  Rows Remaining :({df.shape[0]})")
    return df

# _______Stage 7 — Date Conversion + Tenure___________________________________________________________________
def clean_dates(df, logger):
    logger.info('Stage 7 | Converting Dt_Customer to datetime and computing Tenure')

    df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'])
    # TYPE REASON: Dt_Customer was stored as object (string). Converting to datetime64
    # allows date arithmetic like computing how long a customer has been enrolled.

    df['Tenure_Days']  = (pd.Timestamp.now() - df['Dt_Customer']).dt.days
    df['Tenure_Years'] = df['Tenure_Days'] // 365

    logger.info(f'  Tenure range: {df["Tenure_Days"].min()} – {df["Tenure_Days"].max()} days')
    logger.info('Stage 7 | Date conversion complete')
    return df

# _______Stage 8 — Purchase Channel Outliers_____________________________________________________________________

def clean_purchase_channels(df, logger):
    before = df.shape[0]
    logger.info('Stage 8 | Cleaning purchase channel outliers')

    # --- NumWebPurchases ---
    web_outliers = df[df['NumWebPurchases'] >= 23]
    logger.warning(f'  NumWebPurchases >= 23: {len(web_outliers)} rows')
    df = df[df['NumWebPurchases'] < 23]
    # DROP REASON: all customers with NumWebPurchases >= 23 also have very low income
    # (< $10k), zero store/catalog purchases, and near-zero spending — a pattern
    # inconsistent with legitimate customer behaviour. The next highest value is 11,
    # making 23–27 clear outliers. Multiple suspicious columns confirm these are
    # bad records, not just high-value customers.

    # --- NumCatalogPurchases ---
    cat_outliers = df[df['NumCatalogPurchases'] == 28]
    logger.warning(f'  NumCatalogPurchases == 28: {len(cat_outliers)} rows')
    df = df[df['NumCatalogPurchases'] != 28]
    # DROP REASON: the single row with 28 catalog purchases also has income of $2,447
    # and near-zero spending everywhere else. The max for all other customers is 11.
    # The combination of extremely low income + abnormally high catalog orders is
    # logically inconsistent — this is a data entry error.

    # --- NumWebVisitsMonth ---
    visit_outliers = df[df['NumWebVisitsMonth'] >= 13]
    logger.warning(f'  NumWebVisitsMonth >= 13: {len(visit_outliers)} rows')
    df = df[df['NumWebVisitsMonth'] < 13]
    # DROP REASON: all rows with >= 13 web visits also have very low income (< $10k),
    # zero purchases across all channels, and near-zero spending — visiting the website
    # many times but never buying anything on a very low income makes no sense.
    # The normal range is 0–9 for the vast majority of customers.

    logger.info(f'Stage 8 | Total rows dropped: {before - df.shape[0]}  (rows remaining: {df.shape[0]})')
    return df

# _______Stage 9 — Feature Engineering___________________________________________________________________________________

def feature_engineering(df, logger):
    logger.info('Stage 9 | Feature engineering')

    # Total children at home
    df['Total_Children'] = df['Kidhome'] + df['Teenhome']

    # Total amount spent across all product categories
    spend_cols = ['MntWines', 'MntFruits', 'MntMeatProducts',
                  'MntFishProducts', 'MntSweetProducts', 'MntGoldProds']
    df['Total_Spending'] = df[spend_cols].sum(axis=1)

    # Total purchases across all channels
    purchase_cols = ['NumDealsPurchases', 'NumWebPurchases',
                     'NumCatalogPurchases', 'NumStorePurchases']
    df['Total_Purchases'] = df[purchase_cols].sum(axis=1)

    # Recency segment
    df['Recency_Segment'] = pd.cut(df['Recency'],
                                    bins=[0, 30, 60, 100],
                                    labels=['Active', 'Moderate', 'Inactive'],
                                    include_lowest=True)

    logger.info('  New columns: Total_Children, Total_Spending, Total_Purchases, Recency_Segment')
    logger.info('Stage 9 | Feature engineering complete')
    return df


# _________Stage 10 — Final Validation____________________________________________________________________________

def final_validation(df, logger):
    logger.info('Stage 10 | Final validation')

    nulls = df.isnull().sum()
    if nulls.any():
        logger.warning(f'  Remaining nulls:\n{nulls[nulls > 0]}')
    else:
        logger.info('  No null values remaining')

    dups = df.duplicated().sum()
    logger.info(f'  Duplicate rows remaining: {dups}')
    logger.info(f'  Final shape: {df.shape[0]} rows x {df.shape[1]} columns')
    logger.info(f'  Columns: {list(df.columns)}')
    logger.info('Stage 10 | Validation complete')
    return df

# ── Stage 11 : Database Engine ───────────────────────────────────────

def create_db_engine(server_name, database_name, logger):
    logger.info("Stage 11 | Creating database engine...")
    try:
        params = urllib.parse.quote_plus(
            f'DRIVER={{ODBC Driver 18 for SQL Server}};'
            f'SERVER={server_name};'
            f'DATABASE={database_name};'
            f'Trusted_Connection=yes;'
            f'Encrypt=yes;'
            f'TrustServerCertificate=yes;'
        )
        engine = create_engine(
            f"mssql+pyodbc:///?odbc_connect={params}",
            fast_executemany=True
        )
        logger.info("  Database engine created successfully.")
        return engine
    except Exception as e:
        logger.error(f"  Failed to create engine: {e}")
        raise


#__________Stage 12 : Upload to SQL Server_______________________________________________________________________

def upload_to_sql(df, table_name, engine, unique_keys, dtype_map, logger):
    logger.info(f"Stage 12 | Uploading '{table_name}' - {len(df)} records...")

    try:
        inspector    = inspect(engine)
        table_exists = inspector.has_table(table_name)

        # CASE 1 — Fresh upload
        if not table_exists:
            logger.info(f"  Table '{table_name}' not found - fresh upload...")
            df["uploaded_at"] = pd.Timestamp.now()
            df.to_sql(
                table_name,
                engine,
                if_exists="replace",
                index=False,
                chunksize=10000,
                dtype=dtype_map
            )
            logger.info(f"  Fresh upload complete - {len(df)} records inserted.")

        # CASE 2 — Smart insert (no duplicates)
        else:
            logger.info(f"  Table '{table_name}' exists - checking for duplicates...")

            existing_df = pd.read_sql(
                f"SELECT {', '.join(unique_keys)} FROM {table_name}",
                engine
            )
            logger.info(f"  Existing records in DB: {len(existing_df)}")

            merged   = df.merge(existing_df, on=unique_keys, how="left", indicator=True)
            new_rows = merged[merged["_merge"] == "left_only"].drop(columns="_merge")

            if len(new_rows) == 0:  
                logger.info(f"  No new records for '{table_name}' - upload skipped.")
                return

            new_rows["uploaded_at"] = pd.Timestamp.now()
            new_rows.to_sql(
                table_name,
                engine,
                if_exists="append",
                index=False,
                chunksize=10000,
                dtype=dtype_map
            )
            logger.info(
                f"  Smart upload complete - - "
                f"{len(new_rows)} new records inserted, "
                f"{len(df) - len(new_rows)} duplicates skipped."
            )

        # ── Row Count Validation (NEW) ──────────────────────────────
        db_count = pd.read_sql(f"SELECT COUNT(*) AS cnt FROM {table_name}", engine).iloc[0]['cnt']
        logger.info(f" Row count validation - DB now has {db_count} records in '{table_name}'.")

    except Exception as e:
        logger.error(f"  Upload failed for '{table_name}': {e}")
        raise

# _________Dtype_maps____________________________________________________________________________________________________

DTYPE_MAPS = {
    "supermarket_customers": {
        # ── Identifiers ──
        "Id"                  : Integer(),

        # ── Demographics ──
        "Year_Birth"          : Integer(),
        "Age"                 : Integer(),        # engineered column
        "Education"           : String(50),
        "Marital_Status"      : String(50),
        "Income"              : Float(),
        "Kidhome"             : Integer(),
        "Teenhome"            : Integer(),
        "Total_Children"      : Integer(),        # engineered column

        # ── Enrollment ──
        "Dt_Customer"         : DateTime(),
        "Tenure_Days"         : Integer(),        # engineered column
        "Tenure_Years"        : Integer(),        # engineered column

        # ── Recency ──
        "Recency"             : Integer(),
        "Recency_Segment"     : String(20),       # engineered column

        # ── Spending (amount in currency) ──
        "MntWines"            : Integer(),
        "MntFruits"           : Integer(),
        "MntMeatProducts"     : Integer(),
        "MntFishProducts"     : Integer(),
        "MntSweetProducts"    : Integer(),
        "MntGoldProds"        : Integer(),
        "Total_Spending"      : Integer(),        # engineered column

        # ── Purchase Channels ──
        "NumDealsPurchases"   : Integer(),
        "NumWebPurchases"     : Integer(),
        "NumCatalogPurchases" : Integer(),
        "NumStorePurchases"   : Integer(),
        "Total_Purchases"     : Integer(),        # engineered column
        "NumWebVisitsMonth"   : Integer(),

        # ── Campaign & Engagement ──
        "Response"            : Boolean(),
        "Complain"            : Boolean(),
    }
}

#________ Unique Key _________________________________________________________________________________
UNIQUE_KEYS = {
    "supermarket_customers": ["Id"]
    # Id is the only unique identifier per customer row.
}



 

In [2]:

# ________ Master Pipeline ___________________________________________________________________________

def run_pipeline(file_path, server_name, database_name, log_file="Supermarket_Pipeline.txt"):
    logger = setup_logging(log_file)
    logger.info("=" * 70)
    logger.info("Supermarket Marketing Campaign Pipeline  -  START")
    logger.info("=" * 70)

    try:
        # ── Cleaning Stages ──────────────────────────────────────────────────
        df = ingest(file_path, logger)
        df = remove_duplicates(df, logger)
        df = clean_age(df, logger)
        df = clean_education(df, logger)
        df = clean_Marital_Status(df, logger)
        df = clean_income(df, logger)
        df = clean_dates(df, logger)
        df = clean_purchase_channels(df, logger)
        df = feature_engineering(df, logger)
        df = final_validation(df, logger)

        # ── Database Stage ───────────────────────────────────────────────────
        engine = create_db_engine(server_name, database_name, logger)
        logger.info("Stage 13 | Uploading table to SQL Server")

        upload_to_sql(
            df,
            "supermarket_customers",
            engine,
            UNIQUE_KEYS["supermarket_customers"],
            DTYPE_MAPS["supermarket_customers"],
            logger
        )

        logger.info("=" * 70)
        logger.info("Supermarket Marketing Campaign Pipeline  -  COMPLETE")
        logger.info("=" * 70)
        return df

    except Exception as e:
        logger.critical(f"Pipeline FAILED : {e}", exc_info=True)
        raise


# _________ Run ______________________________________________________________________________________

file_path     = r"D:\DATA SETS (Project)\Capston Project Data\Superstore_Marketing_Campaign_Analysis\Superstore Marketing Campaign Dataset.csv"
server_name   = r"LAPTOP-2VUPGIQA\SQLEXPRESS"
database_name = r"Supermarket_Campaign_DB"

data = run_pipeline(file_path, server_name, database_name, log_file="Supermarket_Pipeline.txt")
data.head()


2026-07-01 10:28:42 | INFO     | UPITransactionPipeline | ======================================================================
2026-07-01 10:28:42 | INFO     | UPITransactionPipeline | Supermarket Marketing Campaign Pipeline  -  START
2026-07-01 10:28:42 | INFO     | UPITransactionPipeline | ======================================================================
2026-07-01 10:28:42 | INFO     | UPITransactionPipeline | Stage 1 | Ingestion Started
2026-07-01 10:28:42 | INFO     | UPITransactionPipeline | Stage 1 | Ingestion Completed -> 2240 rows, 22 columns
2026-07-01 10:28:42 | INFO     | UPITransactionPipeline | Stage 2 | Cheking duplicates 
2026-07-01 10:28:42 | INFO     | UPITransactionPipeline | Stage 2 | Duplicates removed : 0 ( rows remaining 2240)
2026-07-01 10:28:42 | INFO     | UPITransactionPipeline | Stage 3 | Creating Age column and removing outliers
2026-07-01 10:28:42 | WARNING  | UPITransactionPipeline |  Age > 90 found (3 rows) : [1893, 1899, 1900]
2026-07-01 10:28:42

C:\Users\user\AppData\Local\Temp\ipykernel_25500\3250929505.py:155: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  df['Income'].fillna(df['Income'].median() , inplace = True)
C:\Users\user\AppData\Local\Temp\ipykernel_25500\3250929505.py:184: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['Dt_Customer'] = pd.to_datetime(df['Dt_Customer'])


2026-07-01 10:28:43 | INFO     | UPITransactionPipeline |   Tenure range: 4225 – 5288 days
2026-07-01 10:28:43 | INFO     | UPITransactionPipeline | Stage 7 | Date conversion complete
2026-07-01 10:28:43 | INFO     | UPITransactionPipeline | Stage 8 | Cleaning purchase channel outliers
2026-07-01 10:28:43 | WARNING  | UPITransactionPipeline |   NumWebPurchases >= 23: 4 rows
2026-07-01 10:28:43 | WARNING  | UPITransactionPipeline |   NumCatalogPurchases == 28: 1 rows
2026-07-01 10:28:43 | WARNING  | UPITransactionPipeline |   NumWebVisitsMonth >= 13: 9 rows
2026-07-01 10:28:43 | INFO     | UPITransactionPipeline | Stage 8 | Total rows dropped: 14  (rows remaining: 2215)
2026-07-01 10:28:43 | INFO     | UPITransactionPipeline | Stage 9 | Feature engineering
2026-07-01 10:28:43 | INFO     | UPITransactionPipeline |   New columns: Total_Children, Total_Spending, Total_Purchases, Recency_Segment
2026-07-01 10:28:43 | INFO     | UPITransactionPipeline | Stage 9 | Feature engineering complete

C:\Users\user\AppData\Local\Temp\ipykernel_25500\3250929505.py:309: SAWarning: Unrecognized server version info '17.0.1115.1'.  Some SQL Server features may not function properly.
  inspector    = inspect(engine)


2026-07-01 10:28:43 | INFO     | UPITransactionPipeline |   Table 'supermarket_customers' exists - checking for duplicates...
2026-07-01 10:28:43 | INFO     | UPITransactionPipeline |   Existing records in DB: 2215
2026-07-01 10:28:43 | INFO     | UPITransactionPipeline |   No new records for 'supermarket_customers' - upload skipped.
2026-07-01 10:28:43 | INFO     | UPITransactionPipeline | ======================================================================
2026-07-01 10:28:43 | INFO     | UPITransactionPipeline | Supermarket Marketing Campaign Pipeline  -  COMPLETE
2026-07-01 10:28:43 | INFO     | UPITransactionPipeline | ======================================================================


,Id,Year_Birth,Education,Marital_Status,Income,Kidhome,Teenhome,Dt_Customer,Recency,MntWines,...,NumWebVisitsMonth,Response,Complain,Age,Tenure_Days,Tenure_Years,Total_Children,Total_Spending,Total_Purchases,Recency_Segment
0,1826,1970,Graduate,Divorced,84835.0,0,0,2014-06-16,0,189,...,1,1,0,56,4398,12,0,1190,15,Active
1,1,1961,Graduate,Single,57091.0,0,0,2014-06-15,0,464,...,5,1,0,65,4399,12,0,577,18,Active
2,10476,1958,Graduate,Married,67267.0,0,1,2014-05-13,0,134,...,2,0,0,68,4432,12,1,251,11,Active
3,1386,1967,Graduate,Married,32474.0,1,1,2014-11-05,0,10,...,7,0,0,59,4256,11,2,11,4,Active
4,5371,1989,Graduate,Single,21474.0,1,0,2014-08-04,0,6,...,7,1,0,37,4349,11,1,91,8,Active
